In [ ]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

In [ ]:
def user_vectors():
    csv_path = os.path.join("Task 1", "mongodb", "user_vec.parquet")
    checkpoint_path = "user_vectors_checkpoint.npy"

    #I am using gpu since my cpu is bad, delete this and it will auto use cpu
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    #use sentence transformer for semantic
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

    #data
    df = pd.read_parquet(csv_path)
    items = df['item_string'].fillna('').tolist()

    #checkpoint to save progress at each batch
    if os.path.exists(checkpoint_path):
        saved_data = np.load(checkpoint_path)
        all_vectors = list(saved_data)
        start_index = len(all_vectors)
    else:
        all_vectors = []
        start_index = 0


    chunk_size = 5000
    
    #get start row
    start_row = start_index * chunk_size

    for i in range(start_row, len(items), chunk_size):  
        batch = items[i : i + chunk_size]
        
        #encode on GPU
        batch_vecs = model.encode(batch, batch_size=128, convert_to_numpy=True)
        
        #append list
        all_vectors.append(batch_vecs)
        
        #checkpoint at this point
        np.save(checkpoint_path, np.vstack(all_vectors))
        print(f"Saved progress at row {i + len(batch)}")

    #get all vectors together
    vectors = np.vstack(all_vectors)
    
    df_out = pd.DataFrame({
        'id': df['_id'].astype(str),
        'vector': list(vectors),
        'metadata': [{'user_id': str(uid)} for uid in df['_id']]
    })
    
    df_out.to_parquet("master_user_vectors_sentence.parquet", engine="pyarrow")
    
    #remove checkpoint_path since not needed anymore
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        

user_vectors()

In [ ]:
def product_vectors():
    csv_path = os.path.join("Task 1", "mongodb", "product_vec.parquet")
    checkpoint_path = "product_vectors_checkpoint.npy"

    #I am using gpu since my cpu is bad, delete this and it will auto use cpu
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

    #data
    df = pd.read_parquet(csv_path)
    items = df['combined'].fillna('').tolist()

    #checkpoint to save progress at each batch
    if os.path.exists(checkpoint_path):
        saved_data = np.load(checkpoint_path)
        all_vectors = list(saved_data)
        start_index = len(all_vectors)
    else:
        all_vectors = []
        start_index = 0

    chunk_size = 5000
    start_row = start_index * chunk_size
    

    for i in range(start_row, len(items), chunk_size):

        batch = items[i : i + chunk_size]
        #encode on GPU
        batch_vecs = model.encode(batch, batch_size=128, convert_to_numpy=True)
        
        #append list
        all_vectors.append(batch_vecs)
        
        #save to checkpoint
        np.save(checkpoint_path, np.vstack(all_vectors))
        print(f"Saved progress at row {i + len(batch)}")

    #join all vectors
    vectors = np.vstack(all_vectors)

    df_out = pd.DataFrame({
        'id': df['parent_asin'].astype(str),
        'vector': list(vectors),
        'combined': df['combined'],
        "title": df["title"].astype(str)
    })
    
    df_out.to_parquet("master_product_vectors_sentence.parquet", engine="pyarrow")
    
    #remove checkpoint_path
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        


if __name__ == "__main__":
    product_vectors()